# Spotify Tracks Data Cleaning

### 1. Setup & Data Loading

#### 1.1 Path Configuration

In [1]:
import os 

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    DATA_PATH = "/kaggle/input/spotify-tracks-dataset/dataset.csv"
    OUTPUT_PATH = "/kaggle/working/"
else:
    DATA_PATH = "../data/raw/spotify_tracks.csv"
    OUTPUT_PATH = "../data/cleaned/"

#### 1.2 Import required packages

In [2]:
import pandas as pd 
import argparse

from __future__ import annotations 
from pathlib import Path

#### 1.3 Data Loading

In [3]:
df = pd.read_csv(DATA_PATH)

### 2. Data Cleaning

#### 2.1 Missing Values

In [6]:
# Check for missing values, return 1 if original value is missing; 0 otherwise
print("Attributes with missing values:")
print("1 if missing, 0 otherwise")
print()
print(df.isnull().sum())
print()
print("Total number of missing values:")
print(df.isnull().sum().sum())
print()

Attributes with missing values:
1 if missing, 0 otherwise

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

Total number of missing values:
3



In [7]:
# Drop rows with missing values
original_rows = len(df)

df = df.dropna()
print(f"Number of rows before dropping missing values: {original_rows}")
print(f"Number of rows after dropping missing values: {len(df)}")

Number of rows before dropping missing values: 114000
Number of rows after dropping missing values: 113999


#### 2.2 Duplicate Values

In [8]:
# Check for duplicate rows
has_duplicates = df.duplicated().any()
print(f"Duplicate rows: {has_duplicates}")
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Duplicate rows: False
Number of duplicate rows: 0


#### 2.3 Standardise Text Columns

Text Columns: `artists`, `album_name`, `track_name`, `track_genre`

We will do the following to standardise: 

- Remove leading and trailing whitespaces
- Convert to lowercase for consistent matching
- Replace empty strings with NaN if some cells are just space
- Remove multiple spaces inside strings

In [9]:
# Remove leading and trailing whitespace from text columns
text_columns = ["artists", "album_name", "track_name", "track_genre"]
df[text_columns] = df[text_columns].apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [10]:
# Convert to lowercase (useful for `track_genre` & `artists`)
df[text_columns] = df[text_columns].apply(lambda x: x.str.lower() if x.dtype == "object" else x)

In [11]:
# Replace empty strings with NaN if some cells are just space
df[text_columns] = df[text_columns].replace("", pd.NA)


In [12]:
# Remove multiple spaces inside strings
df[text_columns] = df[text_columns].apply(lambda x: x.str.replace(r"\s+", " ", regex=True))

#### 2.4 Handle Categorical Variables

- `explicit` should only contain (_0/1_) and would be useful for one-hot or label encoding, also check for additional values.  

- `track_genre`, `key` & `mode` may also require handling but we will **SKIP** it currently.

In [16]:
print(df["explicit"].value_counts())
print("")
print(df["explicit"].unique())

explicit
False    104252
True       9747
Name: count, dtype: int64

[False  True]


#### 2.5 Check Numerical Columns for Outliers

**Numeric Columns**:   
- `popularity`
- `duration_ms`  
- `tempo`  
- `loudness`  

We need to ensure that there is no negative or impossible values and scale/normalise numeric features for our ML models.

In [17]:
df["duration_ms"].describe()

count    1.139990e+05
mean     2.280312e+05
std      1.072961e+05
min      8.586000e+03
25%      1.740660e+05
50%      2.129060e+05
75%      2.615060e+05
max      5.237295e+06
Name: duration_ms, dtype: float64

In [18]:
df["tempo"].describe()

count    113999.000000
mean        122.147695
std          29.978290
min           0.000000
25%          99.218500
50%         122.017000
75%         140.071000
max         243.372000
Name: tempo, dtype: float64

In [19]:
df["loudness"].describe()

count    113999.000000
mean         -8.258950
std           5.029357
min         -49.531000
25%         -10.013000
50%          -7.004000
75%          -5.003000
max           4.532000
Name: loudness, dtype: float64

In [20]:
df["popularity"].describe()

count    113999.000000
mean         33.238827
std          22.304959
min           0.000000
25%          17.000000
50%          35.000000
75%          50.000000
max         100.000000
Name: popularity, dtype: float64

As seen there is no outliers from above.

#### 2.6 Consistency Checks

We need to check the following:

- `mode` : should be _0/1_
- `time_signature` : should be realistic (usually 3, 4, or 5)

In [21]:
df["mode"].value_counts()

mode
1    72681
0    41318
Name: count, dtype: int64

In [22]:
df["time_signature"].value_counts()

time_signature
4    101842
3      9195
5      1826
1       973
0       163
Name: count, dtype: int64

### 3. Output

In [24]:
df_cleaned = df.copy()

In [25]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
df_cleaned.to_csv(OUTPUT_PATH + "spotify_tracks_cleaned.csv", index=False)
print(f"Cleaned dataset saved to {OUTPUT_PATH}spotify_tracks_cleaned.csv")

Cleaned dataset saved to ../data/cleaned/spotify_tracks_cleaned.csv


You will find the dataset at the `OUTPUT_PATH` specified, next run the next notebook `02_process_spotify.ipynb`.